In [18]:
import pandas as pd
from statsmodels.stats.proportion import proportion_confint
from scipy.stats import chi2_contingency
from scipy.stats import fisher_exact

In [14]:
predictions = pd.read_csv("../data/nq_first_to_100_predictions_full.csv")

predictions.head()

,Date,Day,Bias,Confidence,Auction Direction,Context,Result,Correct,Notes
0,2025-09-01,Monday,Long,57%,Balanced,Balance,Invalid,NaN,Positive ORG is modest and the overnight range...
1,2025-09-02,Tuesday,Short,72%,Strong Down,Trend Continuation,Long,False,"Very large negative ORG, an overnight range we..."
2,2025-09-03,Wednesday,Long,65%,Strong Up,Exhaustion,Long,True,Large positive ORG and a sustained recovery fr...
3,2025-09-04,Thursday,Short,59%,Moderate Down,Balance,Long,False,ORG is modestly positive and the higher-timefr...
4,2025-09-05,Friday,Long,68%,Strong Up,Trend Continuation,Short,False,Large positive ORG and a broad overnight advan...


In [15]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]

weekday_accuracy = (
    predictions
    .dropna(subset=["Correct"])
    .groupby("Day")["Correct"]
    .agg(
        Predictions="count",
        Correct="sum"
    )
    .reindex(weekday_order)
)

weekday_accuracy["Incorrect"] = (
    weekday_accuracy["Predictions"] - weekday_accuracy["Correct"]
)

weekday_accuracy["Accuracy"] = (
    weekday_accuracy["Correct"] / weekday_accuracy["Predictions"] * 100
)

weekday_accuracy

,Predictions,Correct,Incorrect,Accuracy
Day,,,,
Monday,46,23,23,50.0
Tuesday,49,33,16,67.346939
Wednesday,49,29,20,59.183673
Thursday,46,21,25,45.652174
Friday,43,32,11,74.418605


In [16]:
ci_low = []
ci_high = []

for _, row in weekday_accuracy.iterrows():
    low, high = proportion_confint(
        count=row["Correct"],
        nobs=row["Predictions"],
        alpha=0.05,
        method="wilson"
    )

    ci_low.append(low * 100)
    ci_high.append(high * 100)

weekday_accuracy["CI Low"] = ci_low
weekday_accuracy["CI High"] = ci_high

weekday_accuracy.round(2)

,Predictions,Correct,Incorrect,Accuracy,CI Low,CI High
Day,,,,,,
Monday,46,23,23,50.0,36.12,63.88
Tuesday,49,33,16,67.346939,53.38,78.79
Wednesday,49,29,20,59.183673,45.25,71.78
Thursday,46,21,25,45.652174,32.15,59.82
Friday,43,32,11,74.418605,59.76,85.07


In [17]:
contingency = pd.crosstab(
    predictions.loc[predictions["Correct"].notna(), "Day"],
    predictions.loc[predictions["Correct"].notna(), "Correct"]
).reindex(weekday_order)

chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.6f}")

Chi-square statistic: 10.5794
Degrees of freedom: 4
P-value: 0.031721


In [19]:
weekday_comparisons = []

resolved = predictions.dropna(subset=["Correct"]).copy()

for day in weekday_order:
    this_day = resolved[resolved["Day"] == day]
    other_days = resolved[resolved["Day"] != day]

    day_correct = this_day["Correct"].sum()
    day_incorrect = len(this_day) - day_correct

    other_correct = other_days["Correct"].sum()
    other_incorrect = len(other_days) - other_correct

    odds_ratio, p_value = fisher_exact(
        [
            [day_correct, day_incorrect],
            [other_correct, other_incorrect]
        ],
        alternative="two-sided"
    )

    weekday_comparisons.append({
        "Day": day,
        "Day Accuracy": day_correct / len(this_day) * 100,
        "Other Days Accuracy": other_correct / len(other_days) * 100,
        "Difference": (
            day_correct / len(this_day)
            - other_correct / len(other_days)
        ) * 100,
        "P-value": p_value
    })

weekday_comparisons = pd.DataFrame(weekday_comparisons).set_index("Day")

weekday_comparisons.round(3)

,Day Accuracy,Other Days Accuracy,Difference,P-value
Day,,,,
Monday,50.000,61.497,-11.497,0.181
Tuesday,67.347,57.065,10.282,0.252
Wednesday,59.184,59.239,-0.055,1.000
Thursday,45.652,62.567,-16.915,0.045
Friday,74.419,55.789,18.629,0.026
